# 01 · Del origen a RAW: Extraccion Funcional

Este notebook extrae datos de SQL Server (WideWorldImporters) y los guarda en formato RAW con:
- Identificador de lote único
- Timestamp de extracción
- Hash SHA-256 para integridad
- Formato JSON para preservar estructura

**Salida:** Archivos en `datos/bronze/` y tabla `raw.invoice_lines` en destino

In [1]:
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, BD_ORIGEN, BD_DESTINO
from datetime import datetime
import os

# Obtener engines
engine_origen, engine_destino = obtener_engines()

# Rutas de salida
DATOS_DIR = './datos'
BRONZE_DIR = os.path.join(DATOS_DIR, 'bronze')
os.makedirs(BRONZE_DIR, exist_ok=True)

# ID único de lote
LOTE_ID = f"LOTE_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f'Lote: {LOTE_ID}')
print(f'Origen: {BD_ORIGEN}')
print(f'Destino: {BD_DESTINO}')
print(f'Directorio de salida: {BRONZE_DIR}')

✓ Conexión OK (origen): WideWorldImporters
✓ Conexión OK (destino): WideWorldImportersDW2
Lote: LOTE_20261103_112522
Origen: WideWorldImporters
Destino: WideWorldImportersDW2
Directorio de salida: ./datos\bronze


## Paso 1: Crear esquema RAW en destino

In [2]:
from sqlalchemy import text

sql_crear_raw = """
IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'raw')
    EXEC('CREATE SCHEMA raw');

IF OBJECT_ID('raw.invoice_lines', 'U') IS NULL
BEGIN
    CREATE TABLE raw.invoice_lines (
        id_raw                INT PRIMARY KEY IDENTITY(1,1),
        lote_id               NVARCHAR(50) NOT NULL,
        tabla_origen          NVARCHAR(100) NOT NULL,
        numero_fila           INT NOT NULL,
        payload               NVARCHAR(MAX) NOT NULL,
        hash_sha256           NVARCHAR(64) NOT NULL UNIQUE,
        timestamp_extraccion  DATETIME DEFAULT GETDATE(),
        INDEX IX_lote (lote_id),
        INDEX IX_hash (hash_sha256)
    );
END;
"""

try:
    with engine_destino.begin() as conn:
        conn.execute(text(sql_crear_raw))
    print('[OK] Esquema RAW y tabla creada')
except Exception as e:
    print(f'[ERROR] {e}')

[OK] Esquema RAW y tabla creada


## Paso 2: Extraer datos de origen

In [3]:
import pandas as pd
import json
import hashlib

# Extraer invoice lines del origen
consulta = """
SELECT TOP 100 
    InvoiceLineID, InvoiceID, StockItemID, 
    Quantity, UnitPrice, TaxRate, TaxAmount, 
    ExtendedPrice, LastEditedWhen
FROM Sales.InvoiceLines
ORDER BY InvoiceLineID
"""

try:
    df_origen = pd.read_sql_query(consulta, engine_origen)
    print(f'[OK] Se extrajeron {len(df_origen)} registros de Sales.InvoiceLines')
    print(f'Columnas: {list(df_origen.columns)}')
    print(f'\nPrimeros 3 registros:')
    display(df_origen.head(3))
except Exception as e:
    print(f'[ERROR] No se pudo leer origen: {e}')
    df_origen = None

[OK] Se extrajeron 100 registros de Sales.InvoiceLines
Columnas: ['InvoiceLineID', 'InvoiceID', 'StockItemID', 'Quantity', 'UnitPrice', 'TaxRate', 'TaxAmount', 'ExtendedPrice', 'LastEditedWhen']

Primeros 3 registros:


,InvoiceLineID,InvoiceID,StockItemID,Quantity,UnitPrice,TaxRate,TaxAmount,ExtendedPrice,LastEditedWhen
0,1,1,67,10,230.0,15.0,345.00,2645.00,2013-01-01 12:00:00
1,2,2,50,9,13.0,15.0,17.55,134.55,2013-01-01 12:00:00
2,3,2,10,9,32.0,15.0,43.20,331.20,2013-01-01 12:00:00


## Paso 3: Procesar datos en formato RAW

In [4]:
if df_origen is not None and len(df_origen) > 0:
    raw_records = []
    
    for idx, row in df_origen.iterrows():
        # Convertir cada fila a diccionario
        record_dict = row.to_dict()
        # Convertir timestamps a string
        for key, val in record_dict.items():
            if pd.isna(val):
                record_dict[key] = None
            elif hasattr(val, 'isoformat'):
                record_dict[key] = val.isoformat()
            else:
                record_dict[key] = str(val)
        
        # Crear payload JSON ordenado
        payload_json = json.dumps(record_dict, sort_keys=True)
        
        # Calcular hash
        hash_sha256 = hashlib.sha256(payload_json.encode('utf-8')).hexdigest()
        
        raw_records.append({
            'lote_id': LOTE_ID,
            'tabla_origen': 'Sales.InvoiceLines',
            'numero_fila': idx + 1,
            'payload': payload_json,
            'hash_sha256': hash_sha256
        })
    
    df_raw = pd.DataFrame(raw_records)
    print(f'[OK] Se procesaron {len(df_raw)} registros en formato RAW')
    print(f'\nEstructura:')
    print(f'- Lote ID: {LOTE_ID}')
    print(f'- Registros: {len(df_raw)}')
    print(f'\nPrimera fila:')
    display(df_raw.head(1))

[OK] Se procesaron 100 registros en formato RAW

Estructura:
- Lote ID: LOTE_20261103_112522
- Registros: 100

Primera fila:


,lote_id,tabla_origen,numero_fila,payload,hash_sha256
0,LOTE_20261103_112522,Sales.InvoiceLines,1,"{""ExtendedPrice"": ""2645.0"", ""InvoiceID"": ""1"", ...",514646eb56026f38eac922b5e6b141a2fd5b59cf07ca8c...


## Paso 4: Guardar en archivo CSV

In [5]:
if df_raw is not None and len(df_raw) > 0:
    # Archivo CSV con timestamp
    archivo_csv = os.path.join(BRONZE_DIR, f'raw_invoice_lines_{LOTE_ID}.csv')
    
    try:
        df_raw.to_csv(archivo_csv, index=False, encoding='utf-8')
        print(f'[OK] Datos RAW guardados en: {archivo_csv}')
        print(f'Tamaño: {os.path.getsize(archivo_csv):,} bytes')
    except Exception as e:
        print(f'[ERROR] {e}')

[OK] Datos RAW guardados en: ./datos\bronze\raw_invoice_lines_LOTE_20261103_112522.csv
Tamaño: 35,713 bytes


## Paso 5: Insertar en tabla RAW de destino

In [6]:
if df_raw is not None and len(df_raw) > 0:
    try:
        df_raw.to_sql(
            'invoice_lines',
            engine_destino,
            schema='raw',
            if_exists='append',
            index=False,
            chunksize=100
        )
        print(f'[OK] {len(df_raw)} registros insertados en raw.invoice_lines')
    except Exception as e:
        print(f'[ERROR] No se pudo insertar en destino: {e}')

[OK] 100 registros insertados en raw.invoice_lines


## Paso 6: Verificación

In [7]:
from sqlalchemy import text

try:
    # Estadísticas en destino
    sql_stats = """
    SELECT 
        lote_id,
        COUNT(*) as total_registros,
        COUNT(DISTINCT hash_sha256) as hashes_unicos,
        MIN(timestamp_extraccion) as timestamp_inicio,
        MAX(timestamp_extraccion) as timestamp_fin
    FROM raw.invoice_lines
    WHERE lote_id = @lote
    GROUP BY lote_id
    """
    
    with engine_destino.connect() as conn:
        result = conn.execute(text(sql_stats).bindparams(lote=LOTE_ID)).mappings().first()
        
    print('[OK] Verificacion en BD destino:')
    if result:
        for key, val in result.items():
            print(f'  {key}: {val}')
    else:
        print('  Sin resultados (el lote puede haber fallado)')
except Exception as e:
    print(f'[ERROR] {e}')

# Listar archivos en bronze
print(f'\n[OK] Archivos en {BRONZE_DIR}:')
for archivo in os.listdir(BRONZE_DIR):
    ruta = os.path.join(BRONZE_DIR, archivo)
    tamano = os.path.getsize(ruta)
    print(f'  - {archivo} ({tamano:,} bytes)')

[ERROR] This text() construct doesn't define a bound parameter named 'lote'

[OK] Archivos en ./datos\bronze:
  - raw_invoice_lines_LOTE_20261103_112522.csv (35,713 bytes)
